# GRPO on a tiny model: a task it can learn, and a reward hack it finds

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/profrodai/profrodai-resources/blob/main/articles/grpo-on-a-tiny-model/grpo_v1.ipynb)

Companion to https://profrod.ai/articles/grpo-on-a-tiny-model. GRPO written in plain PyTorch (the files are fetched from GitHub in the first cell) trains Qwen2.5-0.5B-Instruct on a task with a programmatic verifier, then trains it again against a verifier with a planted bug to watch the policy exploit it and a detector catch it.

Choose **Runtime > Change runtime type > T4 GPU**, then **Runtime > Run all**. The default size is meant to finish in about 60 minutes on a free T4; each section prints the running total. If the runtime disconnects, run all again: finished runs are skipped (results are kept in `/content/results`, and in Google Drive too if you mount it).

Sections: setup, base eval, GRPO run, eval, the reward hack, receipt.

## Setup

In [ ]:
import json, os, shutil, subprocess, sys, time, urllib.request
T0 = time.time()
def minutes():
    return (time.time() - T0) / 60

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==5.18.0"], check=True)
BASE = "https://raw.githubusercontent.com/profrodai/profrodai-resources/main/articles/grpo-on-a-tiny-model/"
FILES = ["grpo_env_v1.py", "grpo_verify_v1.py", "grpo_stats_v1.py", "grpo_core_v1.py", "grpo_run_v1.py",
         "data/revisions-v1.json", "data/verifier-cases-v1.jsonl", "results/summary-v1.json"]
for name in FILES:
    os.makedirs(os.path.dirname(name) or ".", exist_ok=True)
    if not os.path.exists(name):
        target = name if not name.startswith("results/") else "recorded-summary-v1.json"
        urllib.request.urlretrieve(BASE + name, target)

# Optional: keep results across disconnects in Google Drive.
DRIVE = "/content/drive/MyDrive/grpo-on-a-tiny-model-results"
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as error:
    print("Drive not mounted, results stay in /content/results:", error)
os.makedirs("results", exist_ok=True)
if os.path.isdir(DRIVE):
    shutil.copytree(DRIVE, "results", dirs_exist_ok=True)
    print("restored", len(os.listdir("results")), "result files from Drive")

def checkpoint():
    if os.path.isdir("/content/drive/MyDrive"):
        shutil.copytree("results", DRIVE, dirs_exist_ok=True)

import torch
sys.path.insert(0, os.getcwd())
import grpo_run_v1 as R
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU: choose a T4 runtime")
print(json.dumps(R.environment(), indent=1))

# The configuration. The defaults are sized for a free T4; LARGER is the documented bigger setting.
MODEL, TASK = "qwen05", "units"
TRAIN = dict(model=MODEL, task=TASK, steps=30, lr=2e-05, lora=16, prompts=8, group=8, eval_set="test")
SEEDS = [1, 2, 3, 4, 5]
HACK = dict(model=MODEL, task=TASK, steps=40, lr=2e-05, lora=16, prompts=8, group=8, seed=1, eval_set="test")
LARGER = False  # True: 10 seeds and twice the steps; plan on an L4, or resume across T4 sessions
if LARGER:
    SEEDS, TRAIN["steps"], HACK["steps"] = list(range(1, 11)), 2 * TRAIN["steps"], 2 * HACK["steps"]
MINUTES = {"setup": minutes()}
print(f"setup done at {minutes():.1f} min")

## Base eval

Greedy answers of the untrained model on the held-out items. `strict` is the reward the run trains on (answer in the tags, a plain number, exactly right); `content` ignores the format and asks only whether the last number in the reply is right, so a gain in `strict` alone is a format gain.

In [ ]:
t = minutes()
base = R.base_eval(MODEL, TASK, None, 96, eval_set="test")
print(json.dumps(R.eval_summary(base), indent=1))
MINUTES["baseEval"] = minutes() - t
checkpoint()
print(f"base eval done at {minutes():.1f} min")

## GRPO run

One run per seed. Every step prints the reward, KL to the reference, the policy's entropy, the mean response length, the share of answers the strict parser cannot read, and the share of groups whose rewards were all equal (those groups carry no gradient).

In [ ]:
for seed in SEEDS:
    t = minutes()
    R.train(R.Config(seed=seed, **TRAIN))
    MINUTES[f"train-s{seed}"] = minutes() - t
    checkpoint()
    print(f"seed {seed} done at {minutes():.1f} min")

## Eval

Paired against the base model on the same items: exact McNemar on the items where the two disagree, and a paired bootstrap interval for the change. Then the spread over seeds (a t interval over the per-seed results).

In [ ]:
import grpo_stats_v1 as S
summary = R.summarize()
group = [g for g in summary["groups"] if g.startswith(f"{MODEL}-{TASK}-strict-")]
for name, run in summary["runs"].items():
    if run["config"]["verifier"] != "strict":
        continue
    for k in ("strict", "content"):
        p = run["paired"][k]
        lo, hi = p["diffBootstrap95"]
        print(f"{name} {k:7s}: base {p['base']['rate']:.3f} -> trained {p['trained']['rate']:.3f}  change {p['diff']:+.3f} "
              f"(95% paired bootstrap {lo:+.3f} to {hi:+.3f}; McNemar p = {p['mcnemarP']:.2g}, gained {p['gained']}, lost {p['lost']})")
for g in group:
    for k in ("strict", "content"):
        m, lo, hi = summary["groups"][g][k]["diffMeanT95"]
        print(f"{g} {k}: mean change over {summary['groups'][g]['seeds']} seeds {m:+.3f} (95% t interval {lo:+.3f} to {hi:+.3f})")

## The reward hack

The same run, rewarded by `verify_planted`: its number comparison returns True whenever it cannot parse the answer ("do not punish the policy for the limits of our parser"). The strict verifier still scores the same rollouts in the shadows, and the detector flags the run when the training reward runs more than 0.10 ahead of it for 3 steps in a row.

In [ ]:
t = minutes()
R.train(R.Config(verifier="planted", **HACK))
MINUTES["hack"] = minutes() - t
checkpoint()
summary = R.summarize()
hack = [r for n, r in summary["runs"].items() if r["config"]["verifier"] == "planted"][0]
print("detector:", hack["detector"])
print("first 5 steps:", {k: round(v, 3) for k, v in hack["firstSteps"].items()})
print("last 5 steps: ", {k: round(v, 3) for k, v in hack["lastSteps"].items()})
print("held out, hacked policy: strict", round(hack["paired"]["strict"]["trained"]["rate"], 3),
      "| planted", round(hack["plantedEval"]["rate"], 3), "| unparseable answers", round(hack["unparseableRate"]["trained"], 3))
name = [n for n in summary["runs"] if summary["runs"][n]["config"]["verifier"] == "planted"][0]
rows = R.read_jsonl(R.RESULTS / f"eval-{name}.jsonl")
for r in [r for r in rows if r["unparseable"]][:5]:
    print(repr(r["response"][:120]), "| truth", r["truth"])
print(f"hack done at {minutes():.1f} min")

## Receipt

Device, CUDA, driver, every package version, minutes per run, all results, and the cost ($0: open weights on a free runtime). Compare it with the recorded local run in `recorded-summary-v1.json`.

In [ ]:
MINUTES["total"] = minutes()
rec = R.receipt(R.RESULTS / "receipt-colab-v1.json", {"minutesBySection": MINUTES, "summary": R.summarize()})
checkpoint()
print(json.dumps({k: v for k, v in rec.items() if k != "summary"}, indent=1))
try:
    from google.colab import files
    files.download(str(R.RESULTS / "receipt-colab-v1.json"))
except Exception as error:
    print("download it from the file browser:", error)